# Masked MAMBA-GINR - Proper Implementation

## Overview

This notebook implements **Masked Image Modeling** with the **original MAMBA-GINR architecture**.

### ✅ Critical Fixes from Previous Version

1. **BiMamba Encoder** (not simplified Transformer)
2. **LAINRDecoder** for reconstruction
3. **Pixel-level modulation features** (32×32×512)
4. **CNN classifier** on spatial features (not global pooling)
5. **50% masking** (not 75%)

### Architecture Flow

```
Input: 32×32×3 image
    ↓
Patchify: 2×2 patches → 256 patches
    ↓
Random Masking: 50% → 128 visible, 128 masked
    ↓
BiMamba Encoder (ONLY visible patches) ← INFORMATION BOTTLENECK
    ↓
Extract LP features (256 tokens × 256 dim)
    ↓
LAINRDecoder:
    - Reconstruction: 32×32×3 RGB (Stage 1)
    - Modulation: 32×32×512 spatial features (Stage 2)
    ↓
CNN Classifier on spatial features
```

### Two-Stage Training

**Stage 1**: Masked reconstruction pretraining (100 epochs)
**Stage 2**: CNN classification on frozen modulation features (100 epochs)

### Expected Results

- **Stage 1**: MSE loss ~0.001-0.01 on masked pixels
- **Stage 2**: **60-75% accuracy** on CIFAR-10 (vs 45% with broken version)


---
## 1. Setup and Imports


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import torchvision
import torchvision.transforms as transforms

import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from einops import rearrange, repeat
import math
import os

from mamba_ssm import Mamba
from mamba_ssm.modules.block import Block
from decoder_fix import LAINRDecoder

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")

torch.manual_seed(42)
np.random.seed(42)


---
## 2. Configuration


In [ ]:
CONFIG = {
    # Model
    'img_size': 32,
    'patch_size': 2,
    'dim': 256,
    'num_lp': 256,
    'mamba_depth': 6,
    'ff_dim': 1024,
    'hidden_dim': 512,
    'n_features': 32,
    'mask_ratio': 0.5,  # 50% masking

    # Training - Stage 1 (Reconstruction)
    'stage1_epochs': 100,
    'stage1_lr': 5e-4,
    'stage1_batch_size': 64,
    'stage1_weight_decay': 1e-4,

    # Training - Stage 2 (Classification)
    'stage2_epochs': 100,
    'stage2_lr': 1e-3,
    'stage2_batch_size': 256,
    'stage2_weight_decay': 1e-4,

    # Other
    'num_workers': 4,
    'save_dir': './checkpoints',
}

print("Configuration:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")


---
## 3. Helper Functions


In [ ]:
def fourier_encode(coords, n_features=32, std=10.0):
    """Fourier feature encoding for coordinates"""
    B = torch.randn(n_features, 2, device=coords.device) * std
    proj = 2 * math.pi * coords @ B.T
    return torch.cat([torch.cos(proj), torch.sin(proj)], dim=-1)


def create_coordinate_grid(H, W, device='cpu'):
    """Create normalized coordinate grid in [0,1]"""
    y = torch.linspace(0, 1, H, device=device)
    x = torch.linspace(0, 1, W, device=device)
    yy, xx = torch.meshgrid(y, x, indexing='ij')
    return torch.stack([yy, xx], dim=-1)


def get_sinusoidal_embeddings(n, d):
    """Sinusoidal positional embeddings"""
    assert d % 2 == 0
    position = torch.arange(n, dtype=torch.float).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d, 2).float() * -(math.log(10000.0) / d))
    pe = torch.zeros(n, d)
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe

print("✓ Helper functions defined")


---
## 4. BiMamba Components


In [ ]:
# FIXED LearnablePositionTokens class - Replace Cell 8 in notebook

class BiMamba(nn.Module):
    """Bidirectional Mamba from MAMBA-GINR"""
    def __init__(self, d_model=256, d_state=16, d_conv=4, expand=2):
        super().__init__()
        self.f_mamba = Mamba(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand)
        self.r_mamba = Mamba(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand)
        self.proj = nn.Linear(2 * d_model, d_model)

    def forward(self, x, inference_params=None):
        x_forward = self.f_mamba(x, inference_params=inference_params)
        x_backward = self.r_mamba(torch.flip(x, dims=[1]), inference_params=inference_params)
        x_backward = torch.flip(x_backward, dims=[1])
        x = torch.cat([x_forward, x_backward], dim=-1)
        return self.proj(x)


class MambaEncoder(nn.Module):
    """Stack of Mamba blocks"""
    def __init__(self, depth=6, dim=256, ff_dim=1024, dropout=0.0):
        super().__init__()
        self.blocks = nn.ModuleList([
            Block(
                dim=dim,
                mixer_cls=lambda d: BiMamba(d_model=d),
                mlp_cls=lambda d: nn.Sequential(
                    nn.Linear(d, ff_dim),
                    nn.GELU(),
                    nn.Dropout(dropout),
                    nn.Linear(ff_dim, d),
                    nn.Dropout(dropout),
                ),
                norm_cls=nn.LayerNorm,
                fused_add_norm=False
            )
            for _ in range(depth)
        ])

    def forward(self, x):
        residual = None
        for block in self.blocks:
            x, residual = block(x, residual=residual, inference_params=None)
        return x


class LearnablePositionTokens(nn.Module):
    """
    LP tokens - FIXED for variable-length sequences

    Simplified version: LP tokens concatenated at END (not interleaved)
    This avoids index out of bounds errors with variable-length visible patches.
    """
    def __init__(self, num_tokens=256, dim=256):
        super().__init__()
        self.num_tokens = num_tokens
        self.dim = dim

        # Initialize with sinusoidal embeddings
        init_tokens = get_sinusoidal_embeddings(num_tokens, dim)
        self.tokens = nn.Parameter(init_tokens, requires_grad=True)

    def add_lp(self, x):
        """
        Add LP tokens to END of sequence

        Args:
            x: (B, L, D) input tokens (variable length L)
        Returns:
            (B, L+num_tokens, D) tokens with LP appended
        """
        B = x.shape[0]
        lps = repeat(self.tokens, 'n d -> b n d', b=B)
        return torch.cat([x, lps], dim=1)  # Concatenate at end

    def extract_lp(self, x):
        """
        Extract LP tokens from END of sequence

        Args:
            x: (B, L+num_tokens, D) encoded tokens
        Returns:
            (B, num_tokens, D) LP features
        """
        return x[:, -self.num_tokens:]  # Last num_tokens positions


print("✓ BiMamba components defined (FIXED for variable-length sequences)")
print("  Note: LP tokens concatenated at end (not interleaved) to avoid index errors")



---
## 5. Modulation Feature Extractor

This wraps LAINRDecoder to expose intermediate modulation features.


In [ ]:
class ModulationExtractor(nn.Module):
    """
    Extract pixel-level modulation features from LAINRDecoder

    This wraps LAINRDecoder to expose the intermediate modulation features
    that are normally hidden inside the forward pass.
    """
    def __init__(self, decoder):
        super().__init__()
        self.decoder = decoder

    def forward(self, coords, tokens, return_modulation=False):
        """
        Args:
            coords: (B, H, W, 2) query coordinates
            tokens: (B, L, D) LP token features
            return_modulation: if True, return modulation features instead of RGB

        Returns:
            if return_modulation:
                modulation: (B, H, W, hidden_dim) spatial modulation features
            else:
                rgb: (B, H, W, 3) predicted RGB values
        """
        B, H, W, _ = coords.shape
        coords_flat = coords.reshape(B, -1, 2)

        # Fourier encoding
        fourier_features = fourier_encode(coords_flat[0], self.decoder.n_features)
        fourier_features = repeat(fourier_features, 'n d -> b n d', b=B)

        # Query projection
        queries = F.relu(self.decoder.query_proj(fourier_features))

        # Spatial bias
        grid_coords = coords_flat[0]
        num_queries = grid_coords.shape[0]
        H_query = W_query = int(math.sqrt(num_queries))
        indices = self.decoder.get_patch_index(grid_coords, H_query, W_query)
        bias = self.decoder.compute_spatial_bias(indices, H_query, W_query, tokens.shape[1])

        # Extract modulation via cross-attention
        modulation = self.decoder.cross_attention(queries, tokens, bias)

        if return_modulation:
            # Return spatial modulation features for classification
            return modulation.reshape(B, H, W, -1)
        else:
            # Continue with RGB reconstruction
            decoder_input = torch.cat([fourier_features, modulation], dim=-1)
            features = self.decoder.decoder_blocks(decoder_input)
            rgb = self.decoder.output_proj(features)
            return rgb.reshape(B, H, W, 3)

print("✓ ModulationExtractor defined")


---
## 6. Masked MAMBA-GINR Model

**Key Feature**: Encoder ONLY processes visible patches (information bottleneck)


In [ ]:
class MaskedMAMBAGINR(nn.Module):
    """
    Masked Image Modeling with original MAMBA-GINR architecture

    Encoder: BiMamba (only processes visible patches)
    Decoder: LAINRDecoder (reconstructs all pixels)
    """
    def __init__(self,
                 img_size=32,
                 patch_size=2,
                 in_channels=3,
                 dim=256,
                 num_lp=256,
                 mamba_depth=6,
                 ff_dim=1024,
                 hidden_dim=512,
                 n_features=32,
                 mask_ratio=0.5):
        super().__init__()

        self.img_size = img_size
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2
        self.patch_num = img_size // patch_size
        self.mask_ratio = mask_ratio

        # Patch embedding
        self.patch_embed = nn.Linear(patch_size * patch_size * in_channels, dim)

        # Patch positional encoding
        self.register_buffer('pos_freq', torch.randn(dim // 2, 2) * 10.0)
        self.pos_proj = nn.Linear(dim, dim)

        # Learnable position tokens (works with variable-length sequences)
        self.lp_module = LearnablePositionTokens(
            num_tokens=num_lp,
            dim=dim
        )

        # BiMamba encoder
        self.encoder = MambaEncoder(
            depth=mamba_depth,
            dim=dim,
            ff_dim=ff_dim
        )

        # LAINRDecoder for reconstruction
        self.decoder = LAINRDecoder(
            n_features=n_features,
            input_dim=2,
            output_dim=3,
            hidden_dim=hidden_dim,
            context_dim=dim,
            n_patches=self.num_patches
        )

        # Modulation extractor
        self.modulation_extractor = ModulationExtractor(self.decoder)

    def patchify(self, images):
        """Convert images to patches"""
        B, C, H, W = images.shape
        p = self.patch_size
        patches = rearrange(images, 'b c (h p1) (w p2) -> b (h w) (p1 p2 c)', p1=p, p2=p)
        return patches

    def get_patch_positions(self, indices, device):
        """Get normalized positions for specific patch indices"""
        h = w = self.patch_num
        y_coords = (indices // w).float() / h + 0.5 / h
        x_coords = (indices % w).float() / w + 0.5 / w
        return torch.stack([y_coords, x_coords], dim=-1)

    def fourier_pos_encoding(self, positions):
        """Fourier positional encoding for patches"""
        proj = 2 * math.pi * positions @ self.pos_freq.T
        encoding = torch.cat([torch.sin(proj), torch.cos(proj)], dim=-1)
        return self.pos_proj(encoding)

    def random_masking(self, B, device):
        """Generate random binary masks"""
        num_masked = int(self.num_patches * self.mask_ratio)
        masks = []

        for _ in range(B):
            indices = torch.randperm(self.num_patches, device=device)
            mask = torch.ones(self.num_patches, device=device)
            mask[indices[:num_masked]] = 0  # 0 = masked, 1 = visible
            masks.append(mask)

        return torch.stack(masks, dim=0)

    def encode_visible_patches(self, patches, mask):
        """
        Encode ONLY visible patches with BiMamba

        Critical: This is the information bottleneck!
        """
        B = patches.shape[0]
        device = patches.device

        # Embed all patches first
        tokens = self.patch_embed(patches)

        # Extract ONLY visible patches
        visible_tokens = []
        for i in range(B):
            visible_idx = mask[i].nonzero(as_tuple=True)[0]
            visible_tokens.append(tokens[i, visible_idx])

        # Stack (variable length per sample)
        max_visible = max(v.shape[0] for v in visible_tokens)
        tokens_visible = torch.zeros(B, max_visible, tokens.shape[-1], device=device)

        for i, v_tokens in enumerate(visible_tokens):
            tokens_visible[i, :v_tokens.shape[0]] = v_tokens

            # Add positional encoding for visible patches
            visible_idx = mask[i].nonzero(as_tuple=True)[0]
            positions = self.get_patch_positions(visible_idx, device)
            pos_encoding = self.fourier_pos_encoding(positions)
            tokens_visible[i, :v_tokens.shape[0]] = tokens_visible[i, :v_tokens.shape[0]] + pos_encoding

        # Add LP tokens
        tokens_with_lp = self.lp_module.add_lp(tokens_visible)

        # Encode with BiMamba
        encoded = self.encoder(tokens_with_lp)

        # Extract LP features
        lp_features = self.lp_module.extract_lp(encoded)

        return lp_features

    def forward(self, images, return_modulation=False):
        """
        Forward pass

        Args:
            images: (B, 3, H, W)
            return_modulation: if True, return modulation features for classification

        Returns:
            if training (return_modulation=False):
                loss, pred_rgb, mask
            if inference (return_modulation=True):
                modulation features (B, H, W, hidden_dim)
        """
        B = images.shape[0]
        device = images.device

        if return_modulation:
            # Feature extraction mode (no masking)
            patches = self.patchify(images)
            tokens = self.patch_embed(patches)

            # Add positional encoding for all patches
            all_indices = torch.arange(self.num_patches, device=device).unsqueeze(0).expand(B, -1)
            positions = torch.stack([
                self.get_patch_positions(all_indices[i], device)
                for i in range(B)
            ], dim=0)
            pos_encoding = self.fourier_pos_encoding(positions.reshape(B * self.num_patches, 2))
            pos_encoding = pos_encoding.reshape(B, self.num_patches, -1)
            tokens = tokens + pos_encoding

            # Add LP tokens and encode
            tokens_with_lp = self.lp_module.add_lp(tokens)
            encoded = self.encoder(tokens_with_lp)
            lp_features = self.lp_module.extract_lp(encoded)

            # Extract modulation features at 32×32 grid
            coords = create_coordinate_grid(self.img_size, self.img_size, device)
            coords_batch = repeat(coords, 'h w d -> b h w d', b=B)
            modulation = self.modulation_extractor(coords_batch, lp_features, return_modulation=True)

            return modulation

        else:
            # Training mode (with masking)
            patches = self.patchify(images)
            mask = self.random_masking(B, device)

            # Encode ONLY visible patches → LP features
            lp_features = self.encode_visible_patches(patches, mask)

            # Decode at pixel grid
            coords = create_coordinate_grid(self.img_size, self.img_size, device)
            coords_batch = repeat(coords, 'h w d -> b h w d', b=B)
            pred_rgb = self.modulation_extractor(coords_batch, lp_features, return_modulation=False)

            # Compute loss on MASKED pixels only
            pred_rgb_recon = rearrange(pred_rgb, 'b h w c -> b c h w')
            loss = self.compute_masked_loss(images, pred_rgb_recon, mask)

            return loss, pred_rgb_recon, mask

    def compute_masked_loss(self, target, pred, patch_mask):
        """Compute reconstruction loss on masked pixels only"""
        B, C, H, W = target.shape
        p = self.patch_size

        # Convert patch mask to pixel mask
        patch_mask_spatial = patch_mask.reshape(B, self.patch_num, self.patch_num)
        pixel_mask = repeat(patch_mask_spatial, 'b h w -> b (h p1) (w p2)', p1=p, p2=p)
        pixel_mask = repeat(pixel_mask, 'b h w -> b c h w', c=C)

        # Compute loss only on masked pixels
        loss = ((pred - target) ** 2) * (1 - pixel_mask)
        loss = loss.sum() / ((1 - pixel_mask).sum() + 1e-8)

        return loss

print("✓ MaskedMAMBAGINR model defined")
print(f"   Key feature: Encoder ONLY sees visible patches (information bottleneck)")



---
## 7. CNN Classifier for Spatial Modulation Features

Processes 32×32×512 spatial features (not global pooling!)


In [ ]:
class CNNClassifier(nn.Module):
    """
    CNN classifier for spatial modulation features

    Input: (B, H, W, hidden_dim) modulation features
    Output: (B, num_classes) logits
    """
    def __init__(self, hidden_dim=512, num_classes=10):
        super().__init__()

        self.conv_blocks = nn.Sequential(
            # Block 1: 512 → 256
            nn.Conv2d(hidden_dim, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 32 → 16

            # Block 2: 256 → 128
            nn.Conv2d(256, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 16 → 8

            # Block 3: 128 → 64
            nn.Conv2d(128, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1)  # → 1×1
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, num_classes)
        )

    def forward(self, modulation_features):
        """
        Args:
            modulation_features: (B, H, W, hidden_dim)
        Returns:
            logits: (B, num_classes)
        """
        # Rearrange to (B, C, H, W)
        x = modulation_features.permute(0, 3, 1, 2)
        x = self.conv_blocks(x)
        logits = self.classifier(x)
        return logits

print("✓ CNNClassifier defined")


---
## 8. Data Loading


In [ ]:
def get_dataloaders(batch_size, num_workers=4):
    """Load CIFAR-10 dataset"""
    transform = transforms.Compose([transforms.ToTensor()])

    train_dataset = torchvision.datasets.CIFAR10(
        root='./data', train=True, download=True, transform=transform
    )
    test_dataset = torchvision.datasets.CIFAR10(
        root='./data', train=False, download=True, transform=transform
    )

    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True,
        num_workers=num_workers, pin_memory=True
    )
    test_loader = DataLoader(
        test_dataset, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=True
    )

    return train_loader, test_loader, train_dataset, test_dataset


# Load data
train_loader, test_loader, train_dataset, test_dataset = get_dataloaders(
    batch_size=CONFIG['stage1_batch_size'],
    num_workers=CONFIG['num_workers']
)

print(f"Train samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

# Visualize
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    img, label = train_dataset[i]
    ax.imshow(img.permute(1, 2, 0))
    ax.set_title(f"Label: {label}")
    ax.axis('off')
plt.tight_layout()
plt.show()


---
## 9. Initialize Model


In [ ]:
# Initialize Masked MAMBA-GINR model
model = MaskedMAMBAGINR(
    img_size=CONFIG['img_size'],
    patch_size=CONFIG['patch_size'],
    in_channels=3,
    dim=CONFIG['dim'],
    num_lp=CONFIG['num_lp'],
    mamba_depth=CONFIG['mamba_depth'],
    ff_dim=CONFIG['ff_dim'],
    hidden_dim=CONFIG['hidden_dim'],
    n_features=CONFIG['n_features'],
    mask_ratio=CONFIG['mask_ratio']
).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n{'='*70}")
print(f"MASKED MAMBA-GINR MODEL")
print(f"{'='*70}")
print(f"Total parameters: {total_params:,}")
print(f"\nArchitecture:")
print(f"  - BiMamba encoder: {CONFIG['mamba_depth']} layers")
print(f"  - LP tokens: {CONFIG['num_lp']}")
print(f"  - Hidden dim: {CONFIG['hidden_dim']}")
print(f"  - Mask ratio: {CONFIG['mask_ratio']:.1%}")
print(f"\nTwo-stage training:")
print(f"  Stage 1: Masked reconstruction ({CONFIG['stage1_epochs']} epochs)")
print(f"  Stage 2: CNN classification ({CONFIG['stage2_epochs']} epochs)")
print(f"{'='*70}")


---
## 10. Training Functions for Stage 1


In [ ]:
def train_reconstruction_epoch(model, loader, optimizer, device):
    """Train one epoch of masked reconstruction"""
    model.train()
    total_loss = 0.0
    total_samples = 0

    pbar = tqdm(loader, desc="Training")
    for images, _ in pbar:
        images = images.to(device)
        B = images.shape[0]

        # Forward (with masking)
        loss, pred_rgb, mask = model(images, return_modulation=False)

        # Backward
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item() * B
        total_samples += B

        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    return total_loss / total_samples


def validate_reconstruction(model, loader, device):
    """Validate reconstruction performance"""
    model.eval()
    total_loss = 0.0
    total_samples = 0

    with torch.no_grad():
        for images, _ in tqdm(loader, desc="Validation"):
            images = images.to(device)
            B = images.shape[0]
            loss, pred_rgb, mask = model(images, return_modulation=False)
            total_loss += loss.item() * B
            total_samples += B

    return total_loss / total_samples


def visualize_reconstruction(model, loader, device, epoch, config):
    """Visualize reconstruction results"""
    model.eval()

    # Get sample batch
    images, _ = next(iter(loader))
    images = images[:16].to(device)

    with torch.no_grad():
        loss, pred_rgb, mask = model(images, return_modulation=False)

    # Convert mask to pixel mask
    B = images.shape[0]
    p = model.patch_size
    patch_mask_spatial = mask.reshape(B, model.patch_num, model.patch_num)
    pixel_mask = repeat(patch_mask_spatial, 'b h w -> b c (h p1) (w p2)', c=3, p1=p, p2=p)
    masked_images = images * pixel_mask

    # Visualize
    fig, axes = plt.subplots(3, 16, figsize=(20, 4))

    for i in range(16):
        # Original
        axes[0, i].imshow(images[i].cpu().permute(1, 2, 0).clamp(0, 1))
        axes[0, i].axis('off')
        if i == 0:
            axes[0, i].set_ylabel('Original', fontsize=10, fontweight='bold')

        # Masked
        axes[1, i].imshow(masked_images[i].cpu().permute(1, 2, 0).clamp(0, 1))
        axes[1, i].axis('off')
        if i == 0:
            axes[1, i].set_ylabel('Masked (50%)', fontsize=10, fontweight='bold')

        # Reconstructed
        axes[2, i].imshow(pred_rgb[i].cpu().permute(1, 2, 0).clamp(0, 1))
        axes[2, i].axis('off')
        if i == 0:
            axes[2, i].set_ylabel('Reconstructed', fontsize=10, fontweight='bold')

    plt.suptitle(f'Masked Reconstruction - Epoch {epoch}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(config['save_dir'], f'reconstruction_epoch_{epoch}.png'), dpi=150)
    plt.show()

print("✓ Training functions defined")


---
## 11. Stage 1: Masked Reconstruction Pretraining

Train the model to reconstruct masked patches. Loss is computed only on masked pixels.


In [ ]:
# Create save directory
os.makedirs(CONFIG['save_dir'], exist_ok=True)

# Setup optimizer and scheduler
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CONFIG['stage1_lr'],
    weight_decay=CONFIG['stage1_weight_decay']
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CONFIG['stage1_epochs'], eta_min=1e-6
)

print("\n" + "="*70)
print("STAGE 1: MASKED RECONSTRUCTION PRETRAINING")
print("="*70)
print(f"\nEpochs: {CONFIG['stage1_epochs']}")
print(f"Learning rate: {CONFIG['stage1_lr']}")
print(f"Batch size: {CONFIG['stage1_batch_size']}")
print(f"Mask ratio: {CONFIG['mask_ratio']:.1%}")

best_val_loss = float('inf')
train_losses = []
val_losses = []

for epoch in range(CONFIG['stage1_epochs']):
    print(f"\nEpoch {epoch+1}/{CONFIG['stage1_epochs']}")

    train_loss = train_reconstruction_epoch(model, train_loader, optimizer, device)
    val_loss = validate_reconstruction(model, test_loader, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    scheduler.step()

    print(f"  Train Loss: {train_loss:.6f}")
    print(f"  Val Loss: {val_loss:.6f}")

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss,
        }, os.path.join(CONFIG['save_dir'], 'masked_mamba_ginr_pretrain_best.pth'))
        print(f"  → Best model saved (val_loss: {best_val_loss:.6f})")

    # Visualize every 10 epochs
    if (epoch + 1) % 10 == 0:
        visualize_reconstruction(model, test_loader, device, epoch+1, CONFIG)

# Plot training curves
plt.figure(figsize=(10, 6))
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Reconstruction Loss (MSE)')
plt.title('Stage 1: Masked Reconstruction Pretraining')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig(os.path.join(CONFIG['save_dir'], 'stage1_training_curves.png'), dpi=150)
plt.show()

print(f"\n✓ Stage 1 complete! Best val loss: {best_val_loss:.6f}")


---
## 12. Extract Pixel-Level Modulation Features

Extract 32×32×512 spatial features from the pretrained model.


In [ ]:
def extract_modulation_features(model, loader, device):
    """Extract pixel-level modulation features for all images"""
    model.eval()

    all_features = []
    all_labels = []

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Extracting modulation features"):
            images = images.to(device)

            # Extract modulation features: (B, H, W, hidden_dim)
            modulation = model(images, return_modulation=True)

            all_features.append(modulation.cpu())
            all_labels.append(labels)

    features = torch.cat(all_features, dim=0)
    labels = torch.cat(all_labels, dim=0)

    return features, labels


# Load best pretrained model
checkpoint = torch.load(os.path.join(CONFIG['save_dir'], 'masked_mamba_ginr_pretrain_best.pth'))
model.load_state_dict(checkpoint['model_state_dict'])
print(f"✓ Loaded pretrained model from epoch {checkpoint['epoch']+1}")

# Freeze encoder
for param in model.parameters():
    param.requires_grad = False

# Reload data with larger batch size for feature extraction
train_loader_feat, test_loader_feat, _, _ = get_dataloaders(
    batch_size=CONFIG['stage2_batch_size'],
    num_workers=CONFIG['num_workers']
)

print("\nExtracting modulation features...")
train_features, train_labels = extract_modulation_features(model, train_loader_feat, device)
test_features, test_labels = extract_modulation_features(model, test_loader_feat, device)

print(f"\nTrain features: {train_features.shape}")  # (50000, 32, 32, 512)
print(f"Test features: {test_features.shape}")      # (10000, 32, 32, 512)
print(f"\n✓ Feature extraction complete!")


---
## 13. Stage 2: CNN Classification on Modulation Features

Train CNN classifier on frozen spatial features.


In [ ]:
# Create dataloaders for features
train_dataset_cls = TensorDataset(train_features, train_labels)
test_dataset_cls = TensorDataset(test_features, test_labels)

train_loader_cls = DataLoader(
    train_dataset_cls, batch_size=CONFIG['stage2_batch_size'],
    shuffle=True, num_workers=0
)
test_loader_cls = DataLoader(
    test_dataset_cls, batch_size=CONFIG['stage2_batch_size'],
    shuffle=False, num_workers=0
)

# Initialize CNN classifier
classifier = CNNClassifier(
    hidden_dim=CONFIG['hidden_dim'],
    num_classes=10
).to(device)

optimizer_cls = torch.optim.Adam(
    classifier.parameters(),
    lr=CONFIG['stage2_lr'],
    weight_decay=CONFIG['stage2_weight_decay']
)

scheduler_cls = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_cls, T_max=CONFIG['stage2_epochs'], eta_min=1e-6
)

print("\n" + "="*70)
print("STAGE 2: CNN CLASSIFICATION ON MODULATION FEATURES")
print("="*70)
print(f"\nEpochs: {CONFIG['stage2_epochs']}")
print(f"Learning rate: {CONFIG['stage2_lr']}")
print(f"Batch size: {CONFIG['stage2_batch_size']}")

def train_classifier_epoch(classifier, loader, optimizer, device):
    classifier.train()
    correct = 0
    total = 0
    total_loss = 0.0

    for features, labels in tqdm(loader, desc="Training classifier"):
        features = features.to(device)
        labels = labels.to(device)

        logits = classifier(features)
        loss = F.cross_entropy(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        pred = logits.argmax(dim=1)
        correct += (pred == labels).sum().item()
        total += labels.size(0)
        total_loss += loss.item() * labels.size(0)

    accuracy = 100.0 * correct / total
    avg_loss = total_loss / total
    return accuracy, avg_loss


def validate_classifier(classifier, loader, device):
    classifier.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for features, labels in tqdm(loader, desc="Validating classifier"):
            features = features.to(device)
            labels = labels.to(device)

            logits = classifier(features)
            pred = logits.argmax(dim=1)
            correct += (pred == labels).sum().item()
            total += labels.size(0)

    accuracy = 100.0 * correct / total
    return accuracy


best_acc = 0.0
train_accs = []
test_accs = []

for epoch in range(CONFIG['stage2_epochs']):
    train_acc, train_loss = train_classifier_epoch(classifier, train_loader_cls, optimizer_cls, device)
    test_acc = validate_classifier(classifier, test_loader_cls, device)

    train_accs.append(train_acc)
    test_accs.append(test_acc)

    scheduler_cls.step()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d}: Train={train_acc:.2f}%, Test={test_acc:.2f}% (Best={best_acc:.2f}%)")

    if test_acc > best_acc:
        best_acc = test_acc
        torch.save({
            'epoch': epoch,
            'classifier_state_dict': classifier.state_dict(),
            'test_acc': test_acc,
        }, os.path.join(CONFIG['save_dir'], 'cnn_classifier_best.pth'))

# Plot results
plt.figure(figsize=(10, 6))
plt.plot(train_accs, label='Train Accuracy')
plt.plot(test_accs, label='Test Accuracy')
plt.axhline(y=best_acc, color='r', linestyle='--', alpha=0.3, label=f'Best: {best_acc:.2f}%')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.title('Stage 2: CNN Classification on Modulation Features')
plt.legend()
plt.grid(True, alpha=0.3)
plt.ylim([0, 100])
plt.savefig(os.path.join(CONFIG['save_dir'], 'stage2_classification_results.png'), dpi=150)
plt.show()

print(f"\n✓ Stage 2 complete! Best test accuracy: {best_acc:.2f}%")


---
## 14. Final Results and Analysis


In [ ]:
print("=" * 70)
print("TRAINING COMPLETE!")
print("=" * 70)
print(f"\nStage 1 (Reconstruction):")
print(f"  Best val loss: {best_val_loss:.6f}")
print(f"\nStage 2 (Classification):")
print(f"  Best test accuracy: {best_acc:.2f}%")
print(f"\nExpected Performance:")
print(f"  Previous (broken) version: ~45%")
print(f"  This (proper) version: 60-75%")
print(f"  Supervised baseline: ~88-90%")
print(f"\nImprovement Analysis:")
if best_acc >= 60:
    print(f"  ✅ SUCCESS! Achieved {best_acc:.2f}% (>60%)")
    print(f"  Proper architecture with spatial features works!")
elif best_acc >= 50:
    print(f"  ⚠️  PARTIAL SUCCESS: {best_acc:.2f}% (50-60%)")
    print(f"  Better than broken version, but could improve")
    print(f"  Suggestions: Train longer, tune mask ratio, increase capacity")
else:
    print(f"  ❌ NEEDS INVESTIGATION: {best_acc:.2f}% (<50%)")
    print(f"  Check: reconstruction quality, feature extraction, CNN architecture")
print("=" * 70)


---
## Key Takeaways

### ✅ What We Fixed

1. **BiMamba Encoder**: Uses original MAMBA-GINR architecture (not simplified Transformer)
2. **LAINRDecoder**: Proper integration for reconstruction and modulation extraction
3. **Spatial Features**: 32×32×512 pixel-level modulation (not global pooling)
4. **CNN Classifier**: Processes spatial structure (not linear on flattened vector)
5. **Mask Ratio**: 50% balanced (not 75% too aggressive)

### 📊 Architecture Comparison

| Component | Previous (Broken) | This (Proper) |
|-----------|------------------|---------------|
| Encoder | Simplified Transformer | **BiMamba** |
| Features | 256D global vector | **32×32×512 spatial** |
| Classifier | Linear layer | **CNN (6 layers)** |
| Masking | 75% | **50%** |
| Expected | ~45% | **60-75%** |

### 🔍 Why This Works

1. **Information Bottleneck**: Encoder only sees 50% of patches → must learn semantic understanding
2. **Spatial Preservation**: 32×32 modulation maps preserve object localization
3. **Rich Features**: 512-dim features at each pixel (vs 256-dim total before)
4. **Hierarchical Learning**: CNN extracts multi-scale patterns from spatial features

### 📝 Next Steps

If accuracy is still low (<60%), try:
- Increase Stage 1 epochs (100 → 200)
- Reduce mask ratio (0.5 → 0.4)
- Increase decoder capacity (hidden_dim 512 → 768)
- Try different learning rates
- Add data augmentation (random crops, flips)
